# UK Online Retail — End-to-End Business Analysis
### Alwyn Fernandus | Business Analyst Portfolio Project

---

**Dataset:** UCI Online Retail Dataset (541,909 transactions · Dec 2010 – Dec 2011)  
**Source:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/352/online+retail)  
**Tools:** Python · pandas · matplotlib · seaborn · scikit-learn  

---

## Business Context

This project analyses 13 months of transaction data from a UK-based online gift retailer.
The company sells unique all-occasion gifts, serving both direct consumers and wholesale buyers across 37 countries.

**Business Problem:**  
The company lacks visibility into customer behaviour beyond raw sales figures. It cannot answer: 
- Which customers are about to churn?
- Where is growth actually coming from?
- What products drive disproportionate revenue?
- When and where should the company invest in marketing?

**Project Objective:**  
Build a full analytical picture of revenue health, customer segmentation, product performance, and retention
to support data-driven decisions across marketing, operations, and strategy.

**Key Stakeholders:** Head of E-Commerce, Marketing Manager, CFO, Customer Success

---


## 0. Setup — Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

# ── Design palette (consistent across all charts) ──
PALETTE = {
    'primary'  : '#20808D',
    'secondary': '#A84B2F',
    'dark'     : '#1B474D',
    'light'    : '#BCE2E7',
    'accent'   : '#FFC553',
    'muted'    : '#7A7974',
    'bg'       : '#F7F6F2',
    'text'     : '#28251D',
}
CHART_COLORS = ['#20808D', '#A84B2F', '#1B474D', '#FFC553', '#944454', '#BCE2E7']

plt.rcParams.update({
    'figure.facecolor' : PALETTE['bg'],
    'axes.facecolor'   : '#FFFFFF',
    'axes.edgecolor'   : '#D4D1CA',
    'axes.titlesize'   : 14,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 11,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'grid.color'       : '#E8E7E3',
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.6,
    'figure.dpi'       : 120,
})

print("✓ Libraries loaded and design system configured")

## 1. Load Data

In [ ]:
# Load the dataset
# Source: https://archive.ics.uci.edu/dataset/352/online+retail
df_raw = pd.read_excel('../data/Online Retail.xlsx', engine='openpyxl')

print(f"Shape            : {df_raw.shape}")
print(f"Memory usage     : {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nColumn types:")
print(df_raw.dtypes)
print(f"\nSample data:")
df_raw.head(3)

## 2. Data Cleaning

Before any analysis, the raw data needs to be assessed and cleaned.
The decisions below are documented so they're transparent and reproducible.


In [ ]:
# ── 2.1 Missing Value Assessment ────────────────────────────
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_summary = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
missing_summary = missing_summary[missing_summary['Missing'] > 0]
print("Missing Values:")
print(missing_summary.to_string())

In [ ]:
# ── 2.2 Cleaning Pipeline ────────────────────────────────────

df = df_raw.copy()

# Step 1: Remove rows with no CustomerID (guest/anonymous sessions — not trackable)
df = df.dropna(subset=['CustomerID'])
print(f"After removing null CustomerID : {len(df):,} rows")

# Step 2: Remove cancellations (InvoiceNo starts with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print(f"After removing cancellations   : {len(df):,} rows")

# Step 3: Remove logically invalid rows (negative qty or zero price)
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
print(f"After removing bad qty/price   : {len(df):,} rows")

# Step 4: Remove internal/test stock codes
test_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT']
df = df[~df['StockCode'].isin(test_codes)]
print(f"After removing test codes      : {len(df):,} rows")

# Step 5: Type corrections and feature engineering
df = df.copy()
df['CustomerID']  = df['CustomerID'].astype(int)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Derived features
df['Revenue']    = df['Quantity'] * df['UnitPrice']  # Line-level revenue
df['YearMonth']  = df['InvoiceDate'].dt.to_period('M')
df['Month']      = df['InvoiceDate'].dt.month
df['DayOfWeek']  = df['InvoiceDate'].dt.day_name()
df['Hour']       = df['InvoiceDate'].dt.hour

print(f"\n✓ Final clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
# ── 2.3 Dataset Summary ──────────────────────────────────────
print("="*50)
print("CLEAN DATASET OVERVIEW")
print("="*50)
print(f"  Date range      : {df['InvoiceDate'].min().date()} → {df['InvoiceDate'].max().date()}")
print(f"  Customers       : {df['CustomerID'].nunique():,}")
print(f"  Products        : {df['StockCode'].nunique():,}")
print(f"  Countries       : {df['Country'].nunique()}")
print(f"  Total Revenue   : £{df['Revenue'].sum():,.0f}")
print(f"  Total Orders    : {df['InvoiceNo'].nunique():,}")
df.describe(include='all').T

## 3. KPI Framework

Before diving into the analysis, it's important to define the metrics that matter.
Each KPI below connects directly to a business question.

| KPI | Formula | Why It Matters |
|-----|---------|----------------|
| **Total Revenue** | Sum of (Qty × Unit Price) | Top-line health indicator |
| **Average Order Value (AOV)** | Revenue ÷ Orders | Pricing effectiveness, basket size |
| **Customer Retention Rate** | Repeat Customers ÷ Total Customers | Customer loyalty and product-market fit |
| **Orders per Customer** | Orders ÷ Customers | Engagement depth |
| **RFM Segments** | Recency + Frequency + Monetary scores | Prioritisation of marketing effort |
| **Cohort Retention** | % of cohort returning each month | Structural loyalty vs. churn risk |
| **Revenue Concentration** | Top segment % of revenue | Dependency and business risk |
| **International Revenue Share** | Non-UK Revenue ÷ Total | Global growth momentum |


In [ ]:
# ── KPI Summary Table ────────────────────────────────────────
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
total_revenue   = df['Revenue'].sum()
total_orders    = df['InvoiceNo'].nunique()
total_customers = df['CustomerID'].nunique()
aov             = total_revenue / total_orders
orders_per_cust = total_orders / total_customers
cust_freq       = df.groupby('CustomerID')['InvoiceNo'].nunique()
repeat_pct      = (cust_freq > 1).sum() / total_customers * 100
uk_pct          = df[df['Country']=='United Kingdom']['Revenue'].sum() / total_revenue * 100

kpis = {
    'Metric': ['Total Revenue', 'Total Orders', 'Unique Customers',
               'Avg. Order Value', 'Orders per Customer',
               'Repeat Customer Rate', 'UK Revenue Share'],
    'Value' : [f'£{total_revenue:,.0f}', f'{total_orders:,}', f'{total_customers:,}',
               f'£{aov:.2f}', f'{orders_per_cust:.1f}',
               f'{repeat_pct:.1f}%', f'{uk_pct:.1f}%']
}
pd.DataFrame(kpis).style.set_caption("Business KPI Dashboard")

## 4. Exploratory Data Analysis

### 4.1 Monthly Revenue Trend

In [ ]:
monthly = df.groupby('YearMonth').agg(
    Revenue=('Revenue', 'sum'),
    Orders=('InvoiceNo', 'nunique'),
    Customers=('CustomerID', 'nunique')
).reset_index()
monthly['YearMonth_str'] = monthly['YearMonth'].astype(str)

fig, ax1 = plt.subplots(figsize=(13, 5), facecolor=PALETTE['bg'])
ax2 = ax1.twinx()

ax1.bar(monthly['YearMonth_str'], monthly['Revenue'] / 1000,
        color=PALETTE['primary'], alpha=0.75, width=0.6, label='Revenue (£K)')
ax2.plot(monthly['YearMonth_str'], monthly['Orders'],
         color=PALETTE['secondary'], marker='o', linewidth=2.2,
         markersize=5, label='Orders', zorder=5)

peak_idx = monthly['Revenue'].idxmax()
ax1.annotate(f"Peak\n£{monthly['Revenue'][peak_idx]/1000:.0f}K",
             xy=(peak_idx, monthly['Revenue'][peak_idx]/1000),
             xytext=(peak_idx - 1.5, monthly['Revenue'][peak_idx]/1000 * 0.85),
             arrowprops=dict(arrowstyle='->', color=PALETTE['dark'], lw=1.5),
             fontsize=8.5, color=PALETTE['dark'], fontweight='bold')

ax1.set_ylabel('Revenue (£K)', color=PALETTE['primary'], labelpad=8)
ax2.set_ylabel('Orders', color=PALETTE['secondary'], labelpad=8)
ax1.tick_params(axis='x', rotation=45)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:.0f}K'))
ax1.set_title('Monthly Revenue & Order Volume', fontsize=14, fontweight='bold', pad=12)
ax1.set_facecolor('#FFFFFF')
ax1.grid(axis='y', linestyle='--', alpha=0.5)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nPeak month: {monthly.loc[peak_idx,'YearMonth_str']} — £{monthly['Revenue'][peak_idx]:,.0f}")
print(f"YoY December: Dec 2010 = £{monthly[monthly['YearMonth_str']=='2010-12']['Revenue'].values[0]:,.0f}")

### 4.2 Product Performance

In [ ]:
product_rev = (df.groupby('Description')['Revenue']
               .sum().sort_values(ascending=False)
               .head(10).reset_index())
product_rev['Description'] = product_rev['Description'].str.title().str[:40]
product_rev['Revenue_K'] = product_rev['Revenue'] / 1000

fig, ax = plt.subplots(figsize=(11, 6), facecolor=PALETTE['bg'])
colors = [PALETTE['primary'] if i == 0 else PALETTE['light'] for i in range(len(product_rev))]
bars = ax.barh(product_rev['Description'][::-1], product_rev['Revenue_K'][::-1],
               color=colors[::-1], edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, product_rev['Revenue_K'][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'£{val:.1f}K', va='center', ha='left', fontsize=8.5,
            color=PALETTE['text'], fontweight='bold')
ax.set_xlabel('Total Revenue (£K)')
ax.set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold', pad=12)
ax.set_facecolor('#FFFFFF')
ax.grid(axis='x', linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.set_xlim(0, product_rev['Revenue_K'].max() * 1.22)
plt.tight_layout()
plt.show()

### 4.3 Geographic Distribution

In [ ]:
country_rev = (df[df['Country'] != 'United Kingdom']
               .groupby('Country')['Revenue']
               .sum().sort_values(ascending=False)
               .head(10).reset_index())
country_rev['Revenue_K'] = country_rev['Revenue'] / 1000

fig, ax = plt.subplots(figsize=(11, 6), facecolor=PALETTE['bg'])
bar_colors = [PALETTE['primary']] + [PALETTE['secondary']] * 2 + [PALETTE['light']] * 7
ax.bar(country_rev['Country'], country_rev['Revenue_K'],
       color=bar_colors[:len(country_rev)], edgecolor='white')
for i, (_, row) in enumerate(country_rev.iterrows()):
    ax.text(i, row['Revenue_K'] + 0.5, f'£{row["Revenue_K"]:.1f}K',
            ha='center', va='bottom', fontsize=8.5, fontweight='bold')
ax.set_ylabel('Revenue (£K)')
ax.set_title('International Revenue — Top 10 Countries (excl. UK)', fontsize=14, fontweight='bold', pad=12)
ax.set_facecolor('#FFFFFF')
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### 4.4 Behavioural Patterns — Day & Hour

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_orders = df.groupby('DayOfWeek')['InvoiceNo'].nunique().reindex(dow_order).reset_index()
hour_orders = df.groupby('Hour')['InvoiceNo'].nunique().reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor=PALETTE['bg'])

ax1.bar(day_orders['DayOfWeek'], day_orders['InvoiceNo'],
        color=[PALETTE['primary'] if d not in ['Saturday','Sunday'] else PALETTE['muted']
               for d in dow_order], edgecolor='white')
ax1.set_title('Orders by Day of Week', fontsize=13, fontweight='bold', pad=10)
ax1.tick_params(axis='x', rotation=30)
ax1.set_facecolor('#FFFFFF')
ax1.grid(axis='y', linestyle='--', alpha=0.5)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

ax2.plot(hour_orders['Hour'], hour_orders['InvoiceNo'],
         color=PALETTE['secondary'], linewidth=2.5, marker='o', markersize=5)
ax2.fill_between(hour_orders['Hour'], hour_orders['InvoiceNo'], alpha=0.15, color=PALETTE['secondary'])
peak_hour = hour_orders.loc[hour_orders['InvoiceNo'].idxmax()]
ax2.annotate(f"Peak: {int(peak_hour['Hour'])}:00",
             xy=(peak_hour['Hour'], peak_hour['InvoiceNo']),
             xytext=(peak_hour['Hour']+1.5, peak_hour['InvoiceNo']*0.92),
             arrowprops=dict(arrowstyle='->', color=PALETTE['dark'], lw=1.3),
             fontsize=8.5, fontweight='bold')
ax2.set_title('Orders by Hour of Day', fontsize=13, fontweight='bold', pad=10)
ax2.set_xlabel('Hour (24h)')
ax2.set_facecolor('#FFFFFF')
ax2.grid(axis='y', linestyle='--', alpha=0.5)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Customer Segmentation — RFM Analysis

In [ ]:
# ── Build RFM Table ──────────────────────────────────────────
rfm = df.groupby('CustomerID').agg(
    Recency  =('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary =('Revenue', 'sum')
).reset_index()

# Score each dimension on a 1–4 scale
rfm['R_Score'] = pd.qcut(rfm['Recency'],  4, labels=[4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'),  4, labels=[1, 2, 3, 4])

def segment(row):
    r, f, m = int(row['R_Score']), int(row['F_Score']), int(row['M_Score'])
    score = r + f + m
    if r == 4 and f >= 3 and m >= 3:   return 'Champions'
    elif r >= 3 and f >= 3:             return 'Loyal Customers'
    elif r >= 3 and f <= 2 and m <= 2:  return 'Promising'
    elif r <= 2 and f >= 3 and m >= 3:  return 'At Risk'
    elif r == 1 and f == 1:             return 'Lost'
    elif score >= 9:                    return 'Loyal Customers'
    elif score >= 7:                    return 'Promising'
    elif score >= 5:                    return 'Need Attention'
    else:                               return 'Lost'

rfm['Segment'] = rfm.apply(segment, axis=1)

# Summary
seg_summary = rfm.groupby('Segment').agg(
    Customers =('CustomerID', 'count'),
    Avg_Recency  =('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary =('Monetary', 'mean'),
    Total_Revenue=('Monetary', 'sum')
).round(1).sort_values('Total_Revenue', ascending=False)

seg_summary['Revenue_Share_%'] = (seg_summary['Total_Revenue'] / seg_summary['Total_Revenue'].sum() * 100).round(1)
seg_summary.style.background_gradient(subset=['Total_Revenue'], cmap='YlGnBu').format({
    'Total_Revenue': '£{:,.0f}', 'Revenue_Share_%': '{:.1f}%'
})

In [ ]:
# ── RFM Segment Visualisation ────────────────────────────────
seg_colors = {
    'Champions'       : '#20808D',
    'Loyal Customers' : '#1B474D',
    'Promising'       : '#FFC553',
    'At Risk'         : '#A84B2F',
    'Need Attention'  : '#944454',
    'Lost'            : '#BAB9B4',
}
seg_counts  = rfm['Segment'].value_counts()
seg_revenue = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), facecolor=PALETTE['bg'])
seg_cnt_sorted = seg_counts.sort_values(ascending=True)
ax1.barh(seg_cnt_sorted.index, seg_cnt_sorted.values,
         color=[seg_colors.get(s, PALETTE['muted']) for s in seg_cnt_sorted.index], edgecolor='white')
for i,(s,v) in enumerate(zip(seg_cnt_sorted.index, seg_cnt_sorted.values)):
    ax1.text(v+8, i, f'{v:,}', va='center', fontsize=9, fontweight='bold')
ax1.set_title('Customer Count by Segment', fontsize=13, fontweight='bold', pad=10)
ax1.set_facecolor('#FFFFFF'); ax1.grid(axis='x', linestyle='--', alpha=0.5)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

seg_rev_sorted = seg_revenue.sort_values(ascending=True)
ax2.barh(seg_rev_sorted.index, seg_rev_sorted.values/1000,
         color=[seg_colors.get(s, PALETTE['muted']) for s in seg_rev_sorted.index], edgecolor='white')
for i,(s,v) in enumerate(zip(seg_rev_sorted.index, seg_rev_sorted.values)):
    ax2.text(v/1000+1, i, f'£{v/1000:.0f}K', va='center', fontsize=9, fontweight='bold')
ax2.set_title('Revenue by Segment', fontsize=13, fontweight='bold', pad=10)
ax2.set_facecolor('#FFFFFF'); ax2.grid(axis='x', linestyle='--', alpha=0.5)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

### 5.1 Cohort Retention Analysis

In [ ]:
cohort_df = df[['CustomerID','YearMonth']].copy()
cohort_df['CohortMonth'] = cohort_df.groupby('CustomerID')['YearMonth'].transform('min')
cohort_df['CohortIndex'] = (
    cohort_df['YearMonth'].apply(lambda x: x.ordinal) -
    cohort_df['CohortMonth'].apply(lambda x: x.ordinal)
)
cohort_counts = cohort_df.groupby(['CohortMonth','CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_pivot  = cohort_counts.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')
cohort_size   = cohort_pivot.iloc[:,0]
cohort_pct    = cohort_pivot.divide(cohort_size, axis=0).round(3) * 100
cohort_pct_plot = cohort_pct.iloc[:,:7]
cohort_pct_plot.index = cohort_pct_plot.index.astype(str)

fig, ax = plt.subplots(figsize=(12, 7), facecolor=PALETTE['bg'])
sns.heatmap(cohort_pct_plot, annot=True, fmt='.0f', cmap='YlGnBu',
            linewidths=0.5, linecolor=PALETTE['bg'],
            annot_kws={'size': 9}, ax=ax,
            cbar_kws={'label': 'Retention %', 'shrink': 0.8}, vmin=0, vmax=100)
ax.set_title('Customer Retention Cohort (First 6 Months)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Months After First Purchase', labelpad=8)
ax.set_ylabel('Acquisition Cohort', labelpad=8)
plt.tight_layout()
plt.show()

## 6. Key Business Insights

### Insight 1 — Q4 Revenue Dependency is a Structural Risk
November is the single largest revenue month at 13% of annual revenue.
The business performs well heading into the holiday season, but this concentration
means a bad Q4 (supply chain disruption, competition, economic shock) has
an outsized impact on full-year results.
**Business impact:** A 20% drop in November alone erases ~2.5 months of average revenue.

---

### Insight 2 — Champions Are the Business
Roughly 55% of all revenue comes from a small group of Champions — customers who bought recently,
frequently, and in high amounts. These customers are not being systematically rewarded.
If even 10% of Champions churn, it represents a six-figure annual revenue loss.
**Business impact:** Protecting Champions is the highest-ROI retention activity available.

---

### Insight 3 — 35% of Customers Are One-Time Buyers
34.7% of identifiable customers made exactly one purchase and never returned.
This is a conversion failure at the post-purchase stage — the experience isn't strong
enough to bring people back.
**Business impact:** Converting half of these to repeat buyers would increase customer base by ~17%.

---

### Insight 4 — At-Risk Segment Holds £931K in Revenue
There is a recoverable segment of customers who used to spend heavily
but have gone quiet. The At-Risk segment carries nearly £1M in historical spend —
these are people who proved willingness to pay.
**Business impact:** A re-engagement campaign with 15% success rate = ~£140K recovered.

---

### Insight 5 — International Markets Are Underdeveloped
The UK accounts for 82.9% of revenue. The Netherlands and Ireland are the strongest
international markets, but they are not being invested in proportionally.
EU markets represent a natural geographic expansion opportunity with minimal
logistics overhead given existing supply chains.
**Business impact:** Targeted EU campaigns could realistically add 5–8% to annual revenue.

---

### Insight 6 — Midweek Mornings Are the Highest-Traffic Window
Orders peak on Tuesdays through Thursdays between 10am–2pm.
Weekends are significantly quieter — suggesting B2B or wholesale buyers
rather than weekend retail consumers.
**Business impact:** Marketing spend on weekend digital ads is likely wasted budget.


## 7. Recommendations

### Short-Term (0–3 Months)

| Action | Segment | Expected Impact |
|--------|---------|----------------|
| Launch a Champions loyalty program (early access, exclusive discounts) | Champions | Reduce churn risk on top 55% of revenue |
| Build a 3-email win-back sequence for At-Risk customers | At Risk | Recover £100–150K revenue |
| Shift digital ad budget away from weekends toward Tue–Thu 10am–2pm | All | Improve ROAS by 15–25% |
| Add post-purchase email at Day 7 and Day 30 for new buyers | New Customers | Lift one-time buyer conversion |

---

### Medium-Term (3–9 Months)

- **RFM-driven segmentation in CRM:** Automate segment migration monitoring so marketing triggers
  automatically when a Champion drops to Loyal or At-Risk.
- **November campaign planning:** Start Q4 planning in August. Launch pre-Black Friday
  messaging to existing customers by October 15.
- **Bundle top products:** The top 10 products drive disproportionate revenue.
  Build curated bundles to increase average order value from £476 to £550+.

---

### Strategic (9–18 Months)

- **EU market investment:** Dedicate a small budget to Germany, Netherlands, and France
  with localised campaigns. These markets already buy without marketing investment.
- **Wholesale loyalty program:** Given the B2B buying pattern (weekday peaks, bulk orders),
  create a wholesale tier with volume pricing, account management, and dedicated re-order flows.
- **Demand forecasting model:** Use historical monthly patterns to build a simple ARIMA
  or XGBoost model for inventory planning. November spikes are predictable — stock-outs
  cost revenue and customer trust.


## 8. Conclusion

This analysis converted 13 months of raw transaction logs into a clear strategic picture.

**The headline findings:**
- Revenue is healthy but seasonally concentrated — Q4 is critical and under-planned
- A small group of Champions generates more than half of all revenue — they need protecting
- One-third of customers never return — post-purchase experience needs work
- Nearly £1M in At-Risk revenue is recoverable with targeted outreach
- International expansion (EU) is low-effort, high-upside

The RFM framework provides a repeatable, scalable methodology for prioritising marketing
effort and predicting churn before it happens. With basic CRM automation, these segments
can be maintained monthly without analyst involvement.

---

**Tools used:** Python 3 · pandas · matplotlib · seaborn · scikit-learn  
**Dataset:** [UCI Online Retail Dataset](https://archive.ics.uci.edu/dataset/352/online+retail)  
**Author:** Alwyn Fernandus | MBA Business Analytics | [LinkedIn](https://www.linkedin.com/in/alwynfernandus)
